# Hackathon Dataset Cleaning and Join Analysis

Objective: turn the FDR healthcare-facility data into a cleaner, uncertainty-aware dataset for non-technical planners, medical staff, and researchers.

This notebook follows the hackathon warning from the screenshots: the facility fields are extracted claims from open web text, not verified ground truth. The cleaning step therefore keeps claim evidence, join strategy, confidence, and human-review flags.


## Screenshot-Derived Context

- Source pipeline: web crawl -> GenAI extraction -> entity resolution -> FDR dataset.
- Dataset: roughly 10k India-focused facility records with 51 facility columns.
- Required behavior: cite underlying facility text, communicate uncertainty honestly, and persist review decisions.
- Prior notes: merging records creates uncertainty; useful analysis includes distributions, confidence intervals, Boolean indicators, Bayesian-style confidence, and eventually supervised learning from a reviewed golden set.


In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display

ROOT = Path.cwd()
DATA_DIR = ROOT / "output" / "data"
PLOT_DIR = ROOT / "output" / "plots"

facility_clean = pd.read_csv(DATA_DIR / "facility_health_cleaned.csv")
district_clean = pd.read_csv(DATA_DIR / "district_health_facility_cleaned.csv")
unmatched_pincode_districts = pd.read_csv(DATA_DIR / "district_unmatched_pincode_facility_counts.csv")
summary = json.loads((DATA_DIR / "analysis_summary.json").read_text())
clean = facility_clean  # Alias used by the facility QA cells below.
district_clean.shape, facility_clean.shape, summary["source_tables"]


## Primary Output: District-Level Dataset

District is the safer analytic grain. NFHS health indicators are district-level, India Post lets us bridge PIN codes to district/state, and facility rows are noisy web-extracted claims. The district dataset aggregates facility evidence and quality signals without pretending that each facility record is fully verified.


In [ ]:
district_cols = [
    "state_ut", "district_name", "observed_facility_rows",
    "health_need_score", "district_medical_desert_priority_score",
    "district_data_quality_score", "district_uncertainty_level",
    "source_url_rate", "needs_human_review_rate", "sample_facility_names"
]
district_clean[district_cols].head(20)


## Join Strategy

Facilities do not carry a reliable district field. The defensible join path is:

1. Extract a six-digit Indian PIN from `address_zipOrPostcode`.
2. Join the PIN to India Post.
3. Collapse India Post rows to the modal district/state for each PIN while preserving ambiguity counts.
4. Normalize state and district names.
5. Join to NFHS district health indicators on normalized state + district.
6. Use facility city/state only as a low-confidence fallback when PIN is missing.

The cleaned dataset keeps `join_strategy`, `join_confidence`, `join_match_score`, and `join_uncertainty_reason` so downstream users can filter or review risky rows.


In [ ]:
clean["join_strategy"].value_counts(dropna=False).to_frame("rows")


![Join strategy distribution](../../output/plots/join_strategy_distribution.png)


## Field Coverage and Claim Risk

High field coverage is useful but not equivalent to truth. Description, procedure, equipment, and capability text can be used as evidence snippets, but the scores/rankings should cite those fields and disclose that they are extracted claims.


![Field coverage](../../output/plots/field_coverage.png)


In [ ]:
coverage = pd.DataFrame(summary["field_coverage"]).T
coverage.sort_values("present_pct")


## Distribution Findings

The facility numeric fields are not Gaussian. They are sparse, parsed from text, and heavy-tailed. For ranking and cleaning, the pipeline uses log-scale plots, percentile ranks, and outlier flags instead of deleting high values.


![Numeric distributions](../../output/plots/numeric_distributions_log.png)


In [ ]:
pd.DataFrame(summary["distribution_profiles"]).T


## Geography and Join Uncertainty

Coordinates include off-India values and rows far from their pincode centroid. Those rows are flagged rather than silently dropped because they may reflect extraction errors, entity-resolution mistakes, or facilities with international web artifacts.


![Geo quality scatter](../../output/plots/geo_quality_scatter.png)


In [ ]:
clean["geo_quality"].value_counts(dropna=False).to_frame("rows")


## Medical Desert Proxy

Without true catchment population or verified facility supply, this is a proxy, not a definitive desert label. The score combines:

- NFHS district-level health need percentile.
- Inverse percentile of facility count observed in the joined FDR sample.

Use this to prioritize review and planning questions, not to make final policy claims.


![Medical desert proxy](../../output/plots/medical_desert_proxy_scatter.png)


In [ ]:
cols = [
    "state_ut", "district_name", "observed_facility_rows",
    "health_need_score", "district_medical_desert_priority_score",
    "district_data_quality_score", "district_uncertainty_level",
    "predominant_join_uncertainty", "sample_facility_names"
]
district_clean.sort_values("district_medical_desert_priority_score", ascending=False)[cols].head(20)


## Data Readiness

The readiness score is deliberately conservative. It rewards parseable pincode, health join, plausible coordinates, non-ambiguous PIN bridge, source URLs, claim-text coverage, and contact evidence. Rows below the threshold or with outlier flags are marked `needs_human_review`.


![Data readiness distribution](../../output/plots/data_readiness_distribution.png)


![District data quality distribution](../../output/plots/district_data_quality_distribution.png)


In [ ]:
clean["needs_human_review"].value_counts(dropna=False).to_frame("rows")


## Exported Artifacts

- `output/data/facility_health_cleaned.csv`
- `output/data/district_health_facility_cleaned.csv` (primary cleaned dataset)
- `output/data/district_unmatched_pincode_facility_counts.csv`
- `output/data/facility_health_cleaned_data_dictionary.csv`
- `output/data/district_health_facility_cleaned_data_dictionary.csv`
- `output/data/analysis_summary.json`
- `output/plots/*.png`


In [ ]:
district_path = DATA_DIR / "district_health_facility_cleaned.csv"
facility_audit_path = DATA_DIR / "facility_health_cleaned.csv"
dictionary_path = DATA_DIR / "district_health_facility_cleaned_data_dictionary.csv"
district_path, facility_audit_path, dictionary_path
